In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
import re
from glob import glob
from datetime import datetime


# =========================================================
# INPUT FOLDER
# =========================================================

input_folder = r"Z:\NERDRR\Team_NERDRR_2026\14augrain"


# =========================================================
# OUTPUT FOLDER
# =========================================================

output_folder = r"Z:\NERDRR\Team_NERDRR_2026\14augrain\output\HEM_HOURLY_CSV_single_poit"

os.makedirs(
    output_folder,
    exist_ok=True
)


# =========================================================
# FIND ALL HEM FILES
# =========================================================

all_files = sorted(
    glob(
        os.path.join(
            input_folder,
            "3RIMG_*_L2B_HEM_V01R00.h5"
        )
    )
)


print("=" * 70)
print(f"Total H5 files found : {len(all_files)}")
print("=" * 70)


# =========================================================
# GROUP FILES BY DATE AND HOUR
# =========================================================

hourly_files = {}


for file_path in all_files:

    filename = os.path.basename(file_path)

    match = re.search(
        r"3RIMG_(\d{2}[A-Z]{3}\d{4})_(\d{4})",
        filename
    )

    if match is None:

        print(f"⚠ Skipping invalid filename: {filename}")

        continue


    date_string = match.group(1)
    time_string = match.group(2)


    # Convert 14AUG2026 -> 20260814

    dt = datetime.strptime(
        date_string,
        "%d%b%Y"
    )


    date = dt.strftime(
        "%Y%m%d"
    )


    hour = time_string[:2]


    key = (
        date,
        hour
    )


    if key not in hourly_files:

        hourly_files[key] = []


    hourly_files[key].append(
        file_path
    )


# =========================================================
# PRINT GROUP INFORMATION
# =========================================================

print("\nHourly groups:")

for key in sorted(hourly_files):

    print(
        f"{key[0]} H{key[1]} "
        f"--> {len(hourly_files[key])} files"
    )

# =========================================================
# DOMAIN SUMMARY CSV
# =========================================================

summary_file = os.path.join(
    output_folder,
    "HEM_DOMAIN_HOURLY_RAINFALL.csv"
)

summary_data = []
# =========================================================
# PROCESS EACH HOUR
# =========================================================

for (date, hour) in sorted(hourly_files.keys()):

    files = hourly_files[
        (date, hour)
    ]


    print("\n" + "=" * 70)

    print(
        f"Processing {date} | Hour {hour}"
    )

    print(
        f"Files found : {len(files)}"
    )

    print("=" * 70)


    # =====================================================
    # SORT FILES BY TIME
    # =====================================================

    files = sorted(files)


    # =====================================================
    # OUTPUT DAY FOLDER
    # =====================================================

    day_folder = os.path.join(
        output_folder,
        date
    )


    os.makedirs(
        day_folder,
        exist_ok=True
    )


    # =====================================================
    # OUTPUT FILE
    # =====================================================

    output_file = os.path.join(
        day_folder,
        f"HEM_{date}_H{hour}.csv"
    )


    # =====================================================
    # SKIP EXISTING FILE
    # =====================================================

    if os.path.exists(output_file):

        print(
            f"⏭ Skipping existing file:"
        )

        print(output_file)

        continue


    # =====================================================
    # STORE HEM ARRAYS
    # =====================================================

    hem_list = []

    latitude = None
    longitude = None


    # =====================================================
    # READ ALL FILES IN THIS HOUR
    # =====================================================

    for file_path in files:

        filename = os.path.basename(
            file_path
        )


        print(
            f"Reading: {filename}"
        )


        try:

            with h5py.File(
                file_path,
                "r"
            ) as f:


                # =========================================
                # READ HEM
                # =========================================

                hem = f["HEM"][0, :, :].astype(
                    np.float32
                )


                # =========================================
                # HANDLE FILL VALUE
                # =========================================

                fill_value = f["HEM"].attrs.get(
                    "_FillValue",
                    -999
                )


                hem[
                    hem == float(fill_value[0])
                    if isinstance(fill_value, np.ndarray)
                    else hem == float(fill_value)
                ] = np.nan


                hem_list.append(
                    hem
                )


                # =========================================
                # READ LATITUDE/LONGITUDE ONCE
                # =========================================

                if latitude is None:


                    lat_raw = f[
                        "Latitude"
                    ][:, :].astype(
                        np.float32
                    )


                    lon_raw = f[
                        "Longitude"
                    ][:, :].astype(
                        np.float32
                    )


                    # -------------------------------------
                    # GET ATTRIBUTES
                    # -------------------------------------

                    lat_scale = f[
                        "Latitude"
                    ].attrs.get(
                        "scale_factor",
                        1.0
                    )


                    lon_scale = f[
                        "Longitude"
                    ].attrs.get(
                        "scale_factor",
                        1.0
                    )


                    lat_fill = f[
                        "Latitude"
                    ].attrs.get(
                        "_FillValue",
                        32767
                    )


                    lon_fill = f[
                        "Longitude"
                    ].attrs.get(
                        "_FillValue",
                        32767
                    )


                    # Convert arrays to scalar values

                    lat_scale = float(
                        np.asarray(lat_scale).flat[0]
                    )

                    lon_scale = float(
                        np.asarray(lon_scale).flat[0]
                    )

                    lat_fill = float(
                        np.asarray(lat_fill).flat[0]
                    )

                    lon_fill = float(
                        np.asarray(lon_fill).flat[0]
                    )


                    # -------------------------------------
                    # REMOVE FILL VALUES
                    # -------------------------------------

                    lat_raw[
                        lat_raw == lat_fill
                    ] = np.nan


                    lon_raw[
                        lon_raw == lon_fill
                    ] = np.nan


                    # -------------------------------------
                    # APPLY SCALE FACTOR
                    # -------------------------------------

                    latitude = (
                        lat_raw
                        * lat_scale
                    )


                    longitude = (
                        lon_raw
                        * lon_scale
                    )


        except Exception as e:

            print(
                f"❌ Error reading {filename}"
            )

            print(e)


    # =====================================================
    # CHECK DATA
    # =====================================================

    if len(hem_list) == 0:

        print(
            "⚠ No valid HEM data found."
        )

        continue


    # =====================================================
    # STACK HEM DATA
    # =====================================================

    print(
        "Combining HEM data..."
    )


    hem_stack = np.stack(
        hem_list,
        axis=0
    )


    # =====================================================
    # CALCULATE HOURLY VALUE
    #
    # Average all available observations
    # =====================================================

    hourly_hem = np.nanmean(
        hem_stack,
        axis=0
    )


    # =====================================================
    # PRINT DATA STATISTICS
    # =====================================================

    print(
        f"Hourly HEM minimum: "
        f"{np.nanmin(hourly_hem):.6f}"
    )

    print(
        f"Hourly HEM maximum: "
        f"{np.nanmax(hourly_hem):.6f}"
    )

    print(
        f"Non-zero points: "
        f"{np.sum(hourly_hem > 0):,}"
    )


    # =====================================================
    # VALID MASK
    #
    # Keep valid coordinates and rainfall > 0
    # =====================================================

   # =====================================================
    # REGION
    # =====================================================

    LAT_MIN = 28.18
    LAT_MAX = 28.42

    LON_MIN = 93.75
    LON_MAX = 93.97


    # =====================================================
    # VALID + REGION MASK
    # =====================================================

    valid_mask = (

        ~np.isnan(latitude)

        &

        ~np.isnan(longitude)

        &

        ~np.isnan(hourly_hem)

        &

        (latitude >= LAT_MIN)

        &

        (latitude <= LAT_MAX)

        &

        (longitude >= LON_MIN)

        &

        (longitude <= LON_MAX)

    )

    # =====================================================
    # EXTRACT DOMAIN RAINFALL
    # =====================================================

    domain_rain = hourly_hem[
        valid_mask
    ]


    # =====================================================
    # CALCULATE DOMAIN STATISTICS
    # =====================================================

    domain_mean = np.mean(
        domain_rain
    )

    domain_sum = np.sum(
        domain_rain
    )

    domain_max = np.max(
        domain_rain
    )

    valid_points = len(
        domain_rain
    )


    # =====================================================
    # STORE HOURLY RESULTS
    # =====================================================

    summary_data.append({

        "date": date,

        "hour": int(hour),

        "datetime": f"{date}_{hour}00",

        "valid_grid_points": valid_points,

        "domain_mean_rainfall_mm_hr": domain_mean,

        "domain_sum_rainfall": domain_sum,

        "domain_max_rainfall_mm_hr": domain_max

    })

    # =========================================================
    # SAVE DOMAIN HOURLY SUMMARY
    # =========================================================

    summary_df = pd.DataFrame(
        summary_data
    )

    summary_df.to_csv(
        summary_file,
        index=False,
        float_format="%.6f"
    )


    print("\n============================================")

    print("DOMAIN HOURLY RAINFALL SUMMARY SAVED")

    print(summary_file)

    print("============================================")
    # =====================================================
    # CREATE DATAFRAME
    # =====================================================

    print(
        "Creating DataFrame..."
    )


    df = pd.DataFrame({

        "latitude":

            latitude[
                valid_mask
            ],


        "longitude":

            longitude[
                valid_mask
            ],


        "HEM_hourly":

            hourly_hem[
                valid_mask
            ]

    })


    # =====================================================
    # SAVE CSV
    # =====================================================

    print(
        f"Saving {len(df):,} grid points..."
    )


    df.to_csv(

        output_file,

        index=False,

        float_format="%.6f"

    )


    print(
        f"✅ Saved:"
    )

    print(
        output_file
    )


# =========================================================
# FINISHED
# =========================================================

print("\n" + "=" * 70)

print(
    "🎉 ALL HOURLY HEM FILES CREATED SUCCESSFULLY!"
)

print("=" * 70)